# CrossLS: Surface Code × PQRM Lattice Surgery

**Protocol overview**

CrossLS merges an unrotated surface code patch with a Punctured Quantum Reed-Muller (PQRM) code patch via a ZZ lattice surgery coupler. The key motivation:
- The **surface code** provides efficient, scalable memory and Clifford gates.
- The **PQRM** code supports a transversal non-Clifford gate (T gate and smaller-angle rotations), enabling magic-state injection without distillation.
- The **lattice surgery merge** teleports the logical state between the two codes via d round of ZZ/XX measurements.

## Supported PQRM parameters

| (rx, rz, m) | Code params | Data qubits | Z-stab count | Logical Z weight |
|-------------|-------------|-------------|--------------|------------------|
| (1, 2, 4)   | [[15, 1, 3]] | 15         | 11           | 3                |
| (1, 3, 5)   | [[31, 1, 5]] | 31         | 26           | 7                |
| (1, 4, 6)   | [[63, 1, 7]] | 63         | 57           | 7                |

X stabilizers of PQRM are **post-selected** only (no syndrome ancilla), avoiding the high-weight stabilizer measurements at the cost of a post-selection rate.

## Post-selection modes

- **`pqrm_only`**: post-select on PQRM X-stabilizer violations only.
- **`hybrid`**: post-select on both PQRM X-stabilizer violations and surface code SE errors (stricter, higher rejection rate, lower residual LER).


In [ ]:
import sys
from pathlib import Path
import io, contextlib

ROOT = Path('../..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lightstim.protocols.cross_ls import CrossLSExperiment
from lightstim.noise.config import NoiseConfig

## 1. Build Circuit

In [ ]:
PQRM_para  = [1, 2, 4]   # [[15,1,3]]; try [1,3,5] or [1,4,6]
d_surf     = 3 # try d_surf=3,4,5,6,7 
rounds     = 2 # use rounds=d_surf for better fault-tolerance, here rounds=2 for simpler visualization
PQRM_state = 'Z'   # logical state injected into PQRM side; 'Z', 'X', or 'Y'

exp = CrossLSExperiment(
    PQRM_para=PQRM_para,
    d_surf=d_surf,
    rounds=rounds,
    PQRM_state=PQRM_state,
    surf_state='X',
    if_detector=True,
    post_select_hybrid=True,   # hybrid post-selection (paper setting)
)

with contextlib.redirect_stdout(io.StringIO()):
    circuit = exp.build()

print(f'PQRM{tuple(PQRM_para)}  d_surf={d_surf}  rounds={rounds}  state={PQRM_state}')
print(f'  num_qubits:      {circuit.num_qubits}')
print(f'  num_detectors:   {circuit.num_detectors}')
print(f'  num_observables: {circuit.num_observables}')

## 2. Noiseless Sanity Check

In [ ]:
import numpy as np

sampler = circuit.compile_detector_sampler()
dets, obs = sampler.sample(shots=500, separate_observables=True)
print(f'Noiseless: det errors={np.any(dets)}, obs errors={np.any(obs)}')

## 3. Circuit Diagram

In [ ]:
circuit.diagram('detslice-with-ops-svg')

## 4. Noise Model and Simulation

Noise model: **circuit-level** — p_1q = p_2q = p_meas = p_reset = p.

Post-selection is handled inside `CrossLSExperiment`: the noisy circuit already contains DETECTOR instructions that flag PQRM X-stabilizer violations (and optionally surface SE errors in hybrid mode). The `SimulationPipeline` discards shots where any flagged detector fires.

> **Decoder note**: CrossLS uses **MWPF** (not PyMatching).  
> PQRM X-stabilizers produce high-weight hyperedges in the DEM that PyMatching
> cannot decompose; MWPF handles hyperedges natively.

In [ ]:
from lightstim.simulation.decoder_backend import SimulationPipeline, DecoderConfig

p = 1e-3

exp_noisy = CrossLSExperiment(
    PQRM_para=PQRM_para,
    d_surf=d_surf,
    rounds=rounds,
    PQRM_state=PQRM_state,
    surf_state='X',
    if_detector=True,
    post_select_hybrid=True,
    noise_params=NoiseConfig(p_1q=1e-6, p_2q=p, p_meas=p, p_reset=p),
    noise_model='circuit_level',
)

with contextlib.redirect_stdout(io.StringIO()):
    noisy_circuit = exp_noisy.build()

# Count post-select detectors (those flagging PQRM X-stab violations)
n_det = noisy_circuit.num_detectors
n_obs = noisy_circuit.num_observables
print(f'Noisy circuit: {n_det} detectors, {n_obs} observables')

In [ ]:
from lightstim.simulation.decoder_backend import SimulationPipeline, DecoderConfig

p = 1e-3

exp_noisy = CrossLSExperiment(
    PQRM_para=PQRM_para,
    d_surf=d_surf,
    rounds=rounds,
    PQRM_state=PQRM_state,
    surf_state='X',
    if_detector=True,
    post_select_hybrid=True,
    noise_params=NoiseConfig(p_1q=1e-6, p_2q=p, p_meas=p, p_reset=p),
    noise_model='circuit_level',
)

with contextlib.redirect_stdout(io.StringIO()):
    noisy_circuit = exp_noisy.build()

print(f'Noisy circuit: {noisy_circuit.num_detectors} detectors, {noisy_circuit.num_observables} observables')

pipeline = SimulationPipeline(
    decoder_config=DecoderConfig('mwpf'),
    max_shots=200_000,
    max_errors=50,
    batch_size=10_000,
    num_workers=4,
    print_progress=True,
    progress_interval_sec=5.0,
)

stats = pipeline.run(noisy_circuit)
print(f'\np={p:.0e}  LER={stats.logical_error_rate:.2e}  '
      f'PS_rate={stats.post_selection_rate:.2f}  '
      f'({stats.errors}/{stats.post_selected_shots:,})')

## 5. LER vs PER Sweep (|Z⟩ state)

In [ ]:

import matplotlib.pyplot as plt
from lightstim.plot.styles import apply_paper_style, PALETTE_DISTANCE, bold_ticks
apply_paper_style()

p_values = [5e-4, 1e-3, 2e-3]
distances = [3, 5]   # use [3,5,7] for full paper benchmark

results = {}
for d in distances:
    results[d] = []
    for pval in p_values:
        e = CrossLSExperiment(
            PQRM_para=[1, 2, 4], d_surf=d, rounds=d,
            PQRM_state='Z', surf_state='X',
            if_detector=True, post_select_hybrid=True,
            noise_params=NoiseConfig(p_1q=1e-6, p_2q=pval, p_meas=pval, p_reset=pval),
            noise_model='circuit_level',
        )
        with contextlib.redirect_stdout(io.StringIO()):
            c = e.build()
        pl = SimulationPipeline(
            decoder_config=DecoderConfig('mwpf'),
            max_shots=100_000, max_errors=30,
            batch_size=10_000, num_workers=4, print_progress=False,
        )
        s = pl.run(c)
        results[d].append({'p': pval, 'ler': s.logical_error_rate, 'ps': s.post_selection_rate})
        print(f'd={d} p={pval:.0e}  LER={s.logical_error_rate:.2e}  PS={s.post_selection_rate:.2f}')

fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
for d in distances:
    p_arr = [r['p'] for r in results[d]]
    ler_arr = [r['ler'] for r in results[d]]
    ax.loglog(p_arr, ler_arr, 'o-', color=PALETTE_DISTANCE[d],
              lw=2, ms=7, markeredgecolor='none', label=f'd={d}')

import numpy as np
xlim = ax.get_xlim()
p_ref = np.logspace(np.log10(xlim[0]), np.log10(xlim[1]), 100)
ax.loglog(p_ref, p_ref, 'r--', lw=1.5, label='LER = p')

ax.set_xlabel('Physical Error Rate $p$', fontsize=12)
ax.set_ylabel('LER', fontsize=12)
ax.set_title('CrossLS PQRM(1,2,4) — $|Z\\rangle$ State', fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, which='major', ls='--', alpha=0.5)
bold_ticks(ax)
plt.show()
print('Done.')

## 6. Plot Precomputed Paper Data

For full paper figures (d=3–7, three PQRM codes, bposd GPU decoder), use the precomputed data:

In [ ]:
import pandas as pd

PRECOMP = ROOT / 'paper_artifact' / 'cross_ls' / 'precomputed' / 'all_sweep_data.csv'
df = pd.read_csv(PRECOMP)
df_gpu = df[(df['decoder'] == 'bposd') & (df['backend'] == 'gpu')]

print(f'Loaded {len(df)} rows, {len(df_gpu)} bposd GPU rows')
print(df_gpu[['pqrm','d_surf','state','p_2q','ler','ps_rate']].drop_duplicates()
          .sort_values(['pqrm','state','d_surf','p_2q'])
          .to_string(index=False))